# 01 — Fetch Stock Prices

Pulls OHLCV data + technical indicators (DMA, RSI, MACD, Bollinger Bands, returns) for all Nifty50 tickers via `yfinance`, and upserts into `stock_prices`.

Safe to re-run — `insert_prices()` uses `ON CONFLICT (ticker, date) DO UPDATE`, so re-running never duplicates rows, it just refreshes indicator values.

**Run this first** — the other 3 notebooks depend on `stock_prices` existing.

In [1]:
# This tells Python to look in the parent folder (backend/) for utils/
import sys
sys.path.append('..')

from utils.price_fetcher import fetch_prices, NIFTY50
from utils.db import insert_prices

print("Imports successful")
print(f"Total tickers to fetch: {len(NIFTY50)}")

Imports successful
Total tickers to fetch: 50


## Sanity check — test one ticker before running all 50

In [2]:
prices = fetch_prices("TCS.NS", start="2015-01-01", end="2026-06-30")
print(f"Rows: {len(prices)}")
print("First row:", prices[0])
print("Last row :", prices[-1])

TCS.NS — 2838 rows
Rows: 2838
First row: {'ticker': 'TCS.NS', 'date': datetime.date(2015, 1, 1), 'open_price': 983.8725, 'close_price': 975.6512, 'volume': 366830, 'dma_20': None, 'dma_50': None, 'volume_spike': False, 'rsi': None, 'macd': 0.0, 'macd_signal': 0.0, 'macd_hist': 0.0, 'bb_position': None, 'return_1d': None, 'return_3d': None, 'return_5d': None, 'pct_from_20d_high': None, 'pct_from_20d_low': None}
Last row : {'ticker': 'TCS.NS', 'date': datetime.date(2026, 6, 29), 'open_price': 2090.0, 'close_price': 2097.8999, 'volume': 3412470, 'dma_20': 2168.89, 'dma_50': 2276.9555, 'volume_spike': False, 'rsi': 42.2932, 'macd': -53.2286, 'macd_signal': -51.163, 'macd_hist': -2.0655, 'bb_position': 0.2862, 'return_1d': 0.1528, 'return_3d': -0.5263, 'return_5d': -1.4052, 'pct_from_20d_high': -14.2629, 'pct_from_20d_low': 1.8596}


## Fetch + upsert all Nifty50 tickers

In [4]:
for ticker in NIFTY50:
    prices = fetch_prices(ticker, start="2015-01-01", end="2026-06-30")
    insert_prices(prices)
    print(f"Saved {ticker} — {len(prices)} rows")

print("\nDone — stock_prices populated for all tickers.")

RELIANCE.NS — 2838 rows
Saved RELIANCE.NS — 2838 rows
TCS.NS — 2838 rows
Saved TCS.NS — 2838 rows
HDFCBANK.NS — 2838 rows
Saved HDFCBANK.NS — 2838 rows
INFY.NS — 2838 rows
Saved INFY.NS — 2838 rows
ICICIBANK.NS — 2838 rows
Saved ICICIBANK.NS — 2838 rows
HINDUNILVR.NS — 2838 rows
Saved HINDUNILVR.NS — 2838 rows
ITC.NS — 2838 rows
Saved ITC.NS — 2838 rows
SBIN.NS — 2838 rows
Saved SBIN.NS — 2838 rows
BHARTIARTL.NS — 2838 rows
Saved BHARTIARTL.NS — 2838 rows
KOTAKBANK.NS — 2838 rows
Saved KOTAKBANK.NS — 2838 rows
LT.NS — 2838 rows
Saved LT.NS — 2838 rows
AXISBANK.NS — 2838 rows
Saved AXISBANK.NS — 2838 rows
ASIANPAINT.NS — 2838 rows
Saved ASIANPAINT.NS — 2838 rows
MARUTI.NS — 2838 rows
Saved MARUTI.NS — 2838 rows
SUNPHARMA.NS — 2838 rows
Saved SUNPHARMA.NS — 2838 rows
TITAN.NS — 2838 rows
Saved TITAN.NS — 2838 rows
BAJFINANCE.NS — 2838 rows
Saved BAJFINANCE.NS — 2838 rows
NESTLEIND.NS — 2838 rows
Saved NESTLEIND.NS — 2838 rows
WIPRO.NS — 2838 rows
Saved WIPRO.NS — 2838 rows
ULTRACEMCO.NS 

## Verify

In [5]:
import pandas as pd
from utils.db import get_connection

conn = get_connection()
check = pd.read_sql("""
    SELECT ticker, COUNT(*) AS rows, MIN(date) AS from_date, MAX(date) AS to_date
    FROM stock_prices
    GROUP BY ticker
    ORDER BY ticker
""", conn)
conn.close()

print(f"Total tickers: {len(check)}")
print(check.to_string())

Total tickers: 50
           ticker  rows   from_date     to_date
0     ADANIENT.NS  2838  2015-01-01  2026-06-29
1   ADANIPORTS.NS  2838  2015-01-01  2026-06-29
2   APOLLOHOSP.NS  2838  2015-01-01  2026-06-29
3   ASIANPAINT.NS  2838  2015-01-01  2026-06-29
4     AXISBANK.NS  2838  2015-01-01  2026-06-29
5   BAJAJ-AUTO.NS  2838  2015-01-01  2026-06-29
6   BAJAJFINSV.NS  2838  2015-01-01  2026-06-29
7   BAJFINANCE.NS  2838  2015-01-01  2026-06-29
8   BHARTIARTL.NS  2838  2015-01-01  2026-06-29
9         BPCL.NS  2838  2015-01-01  2026-06-29
10   BRITANNIA.NS  2838  2015-01-01  2026-06-29
11       CIPLA.NS  2838  2015-01-01  2026-06-29
12   COALINDIA.NS  2838  2015-01-01  2026-06-29
13    DIVISLAB.NS  2837  2015-01-01  2026-06-29
14     DRREDDY.NS  2838  2015-01-01  2026-06-29
15   EICHERMOT.NS  2838  2015-01-01  2026-06-29
16      GRASIM.NS  2838  2015-01-01  2026-06-29
17     HCLTECH.NS  2838  2015-01-01  2026-06-29
18    HDFCBANK.NS  2838  2015-01-01  2026-06-29
19    HDFCLIFE.NS  212

C:\Users\acer\AppData\Local\Temp\ipykernel_3844\3947878638.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  check = pd.read_sql("""
